In [3]:
"""
Analysis 3b, Step 1: Download severe (CVSS >= 9.0) CVEs from the NVD API 2.0.

IMPORTANT CONTEXT (read before running)
-----------------------------------------
NIST retired the old bulk JSON data feed files on 2023-12-15. The current,
correct way to get this data is the REST API 2.0 at
https://services.nvd.nist.gov/rest/json/cves/2.0 -- there is no single
downloadable file anymore. This script pages through that API.

Rate limits (as of NVD's current documentation):
  - WITHOUT an API key: 5 requests per rolling 30-second window
  - WITH a free API key: 50 requests per rolling 30-second window
  Get a free key at https://nvd.nist.gov/developers/request-an-api-key
  (note: NVD's key-request page has intermittent maintenance windows --
  if it's down, just run without a key, it'll work, just slower).

The API also caps date-range queries at 120 days per request, and caps
results at 2000 per page. So pulling 2018-2026 (~8 years) requires:
  - ~24 sequential 120-day windows
  - pagination within any window that has >2000 matching CVEs (unlikely
    at CVSS>=9.0, but handled anyway for safety)

This script is resumable: it writes one CSV per date-window-chunk as it
goes, and skips any chunk whose output file already exists. If your
connection drops or NVD rate-limits you mid-run, just re-run the script --
completed chunks won't be re-downloaded.

OUTPUT
------
- nvd_chunks/nvd_chunk_<start>_<end>.csv   (one per 120-day window)
- nvd_severe_cves_2018_2026.csv            (final merged file, after
                                              running step 2 script)
"""

import requests
import pandas as pd
import time
import os
from datetime import datetime, timedelta

# ============================================================
# CONFIG
# ============================================================
API_KEY = None   # <-- paste your free NVD API key here as a string, or leave None
START_DATE = datetime(2018, 1, 1)
END_DATE = datetime(2026, 6, 1)
MIN_CVSS = 9.0
CHUNK_DAYS = 119          # stay safely under the 120-day API limit
OUT_DIR = "nvd_chunks"
BASE_URL = "https://services.nvd.nist.gov/rest/json/cves/2.0"

# Sleep between requests: 6s without a key (5 req / 30s), 0.65s with a key
# (50 req / 30s), per NVD's documented best practice.
SLEEP_SECONDS = 0.65 if API_KEY else 6.0


def fetch_chunk(start, end, min_cvss, max_retries=5):
    """Fetch all CVEs published in [start, end) with CVSS v3 base score
    >= min_cvss, handling pagination within the chunk.

    Transient errors (503 Service Unavailable, connection timeouts) are
    common on NVD's infrastructure and are retried with exponential
    backoff. A 403 (rate limit) is NOT retried automatically here -- it
    raises immediately so the caller can stop and let the rolling rate
    limit window clear before the next manual re-run.
    """
    headers = {"apiKey": API_KEY} if API_KEY else {}
    all_cves = []
    start_index = 0
    results_per_page = 2000

    while True:
        params = {
            "pubStartDate": start.strftime("%Y-%m-%dT%H:%M:%S.000"),
            "pubEndDate": end.strftime("%Y-%m-%dT%H:%M:%S.000"),
            "cvssV3Severity": "CRITICAL",  # pre-filter server-side (CRITICAL = 9.0-10.0 in v3)
            "resultsPerPage": results_per_page,
            "startIndex": start_index,
        }

        resp = None
        last_error = None
        for attempt in range(max_retries):
            try:
                resp = requests.get(BASE_URL, params=params, headers=headers, timeout=60)
            except requests.exceptions.RequestException as e:
                last_error = e
                resp = None

            if resp is not None and resp.status_code == 403:
                raise RuntimeError(
                    "403 Forbidden -- you're likely rate-limited. Wait a few "
                    "minutes and re-run (completed chunks will be skipped)."
                )
            if resp is not None and resp.status_code == 200:
                break

            # Transient failure (503, connection error, etc.) -- back off and retry
            wait = min(60, 5 * (2 ** attempt))
            status = resp.status_code if resp is not None else f"connection error ({last_error})"
            print(f"    Transient error ({status}) on attempt {attempt + 1}/{max_retries}, "
                  f"retrying in {wait}s...")
            time.sleep(wait)
        else:
            # Exhausted retries
            status = resp.status_code if resp is not None else last_error
            raise RuntimeError(
                f"NVD API still failing after {max_retries} retries (last status: {status}). "
                "This chunk was NOT saved -- re-run the script later to resume."
            )

        data = resp.json()
        vulns = data.get("vulnerabilities", [])
        all_cves.extend(vulns)

        total_results = data.get("totalResults", 0)
        start_index += results_per_page
        if start_index >= total_results:
            break
        time.sleep(SLEEP_SECONDS)

    return all_cves


def parse_cve_record(item):
    """Extract the fields we need from one NVD API CVE record."""
    cve = item.get("cve", {})
    cve_id = cve.get("id")
    published = cve.get("published")
    last_modified = cve.get("lastModified")
    source_identifier = cve.get("sourceIdentifier")

    # CVSS v3.1 preferred, fall back to v3.0
    base_score = None
    severity = None
    metrics = cve.get("metrics", {})
    for key in ("cvssMetricV31", "cvssMetricV30"):
        if key in metrics and metrics[key]:
            cvss_data = metrics[key][0].get("cvssData", {})
            base_score = cvss_data.get("baseScore")
            severity = cvss_data.get("baseSeverity")
            break

    return {
        "cve_id": cve_id,
        "published": published,
        "last_modified": last_modified,
        "source_identifier": source_identifier,
        "cvss_base_score": base_score,
        "cvss_severity": severity,
    }


def daterange_chunks(start, end, chunk_days):
    current = start
    while current < end:
        chunk_end = min(current + timedelta(days=chunk_days), end)
        yield current, chunk_end
        current = chunk_end


if __name__ == "__main__":
    os.makedirs(OUT_DIR, exist_ok=True)

    chunks = list(daterange_chunks(START_DATE, END_DATE, CHUNK_DAYS))
    print(f"Will fetch {len(chunks)} date-range chunks of ~{CHUNK_DAYS} days each, "
          f"from {START_DATE.date()} to {END_DATE.date()}.")
    print(f"Using API key: {'YES' if API_KEY else 'NO'} "
          f"(sleep = {SLEEP_SECONDS}s between requests)\n")

    failed_chunks = []
    for i, (chunk_start, chunk_end) in enumerate(chunks, 1):
        fname = os.path.join(
            OUT_DIR,
            f"nvd_chunk_{chunk_start.strftime('%Y%m%d')}_{chunk_end.strftime('%Y%m%d')}.csv"
        )
        if os.path.exists(fname):
            print(f"[{i}/{len(chunks)}] SKIP (already exists): {fname}")
            continue

        print(f"[{i}/{len(chunks)}] Fetching {chunk_start.date()} to {chunk_end.date()} ...")
        try:
            raw_items = fetch_chunk(chunk_start, chunk_end, MIN_CVSS)
        except RuntimeError as e:
            print(f"  ERROR: {e}")
            if "403" in str(e):
                print("  Rate-limited -- stopping the whole run now. Wait a few minutes "
                      "and re-run to resume.")
                break
            print("  Skipping this chunk for now, continuing to the next one. "
                  "Re-run later to fill in this gap.")
            failed_chunks.append((chunk_start.date(), chunk_end.date()))
            time.sleep(SLEEP_SECONDS)
            continue

        records = [parse_cve_record(item) for item in raw_items]
        chunk_df = pd.DataFrame(records)

        # Double-check the score filter client-side too (CRITICAL severity
        # band is 9.0-10.0 in CVSS v3, which matches MIN_CVSS=9.0, but this
        # guards against any edge cases / future CVSS version changes)
        if len(chunk_df) > 0 and "cvss_base_score" in chunk_df.columns:
            chunk_df = chunk_df[chunk_df["cvss_base_score"] >= MIN_CVSS]

        chunk_df.to_csv(fname, index=False)
        print(f"  Saved {len(chunk_df)} CVEs (CVSS >= {MIN_CVSS}) -> {fname}")
        time.sleep(SLEEP_SECONDS)

    print("\nDone with this run. If it stopped early due to rate limiting, "
          "just re-run -- it will resume from the next incomplete chunk.")
    if failed_chunks:
        print(f"\n{len(failed_chunks)} chunk(s) failed after retries and were SKIPPED "
              "(not stopped on, so the rest of the run could continue):")
        for s, e in failed_chunks:
            print(f"  {s} to {e}")
        print("Re-run this script again to retry just these gaps (completed "
              "chunks will be skipped automatically).")
    print("\nOnce all chunks are done, run analysis3b_step2_merge_and_correlate.py "
          "to merge them and run the cross-correlation against your burnout series.")

Will fetch 26 date-range chunks of ~119 days each, from 2018-01-01 to 2026-06-01.
Using API key: NO (sleep = 6.0s between requests)

[1/26] SKIP (already exists): nvd_chunks/nvd_chunk_20180101_20180430.csv
[2/26] Fetching 2018-04-30 to 2018-08-27 ...
  Saved 833 CVEs (CVSS >= 9.0) -> nvd_chunks/nvd_chunk_20180430_20180827.csv
[3/26] Fetching 2018-08-27 to 2018-12-24 ...
    Transient error (connection error (HTTPSConnectionPool(host='services.nvd.nist.gov', port=443): Read timed out. (read timeout=60))) on attempt 1/5, retrying in 5s...
    Transient error (connection error (HTTPSConnectionPool(host='services.nvd.nist.gov', port=443): Read timed out. (read timeout=60))) on attempt 2/5, retrying in 10s...
    Transient error (503) on attempt 3/5, retrying in 20s...
    Transient error (503) on attempt 4/5, retrying in 40s...
  Saved 649 CVEs (CVSS >= 9.0) -> nvd_chunks/nvd_chunk_20180827_20181224.csv
[4/26] Fetching 2018-12-24 to 2019-04-22 ...
    Transient error (503) on attempt 1/5, 